In [25]:
raw_txt = """Service: Médecine interne  |  Consultation du 17/02/2026 à 09:15
Patient: Mme MARTIN Claire, née le 03/11/1978 (47 ans)
N° dossier:  2026-001928   Tel: 06 12 34 56 78    Email: claire.martin78@exemple.fr

Motif: asthénie + dyspnée d’effort, évoluant depuis ~10 jours.
Contexte: HTA connue; diabète type 2 (metformine). Allergies: NKA.
Traitements habituels: Metformine 850 mg x2/j; Ramipril 5mg 1 cp/j.

Examen clinique:
- TA: 152/96 mmHg   FC: 104/min   SpO2: 94% AA   T°: 38,2°C
- Auscultation: râles crépitants bases bilatérales; pas de sibilants.
- Oedèmes MI: +/-
- Douleur thoracique: non; toux: oui (productive).

Biologie (17/02/2026, 10:40):
CRP  86 mg/L;  Leuco  12,4 G/L; Hb 11.2 g/dL; Plaquettes  410 G/L
Créat  98 µmol/L;  Na  134 mmol/L;  K  3,6 mmol/L;  Glycémie  1,42 g/L
Troponine hs:  9 ng/L (N<14)  |  D-dimères:  780 ng/mL

Imagerie:
RX thorax: opacités alvéolo-interstitielles diffuses (à discuter OAP vs infection).
ECG: tachy sinusale, pas de sus-décalage ST.

Conclusion/Plan:
Hypothèse principale: pneumopathie communautaire; ddx: OAP sur poussée HTA.
=> Amoxicilline-ac. clavulanique 1g x3/j 7 jours
=> Paracétamol si T° > 38°C
=> Contrôle clinique à 48h; si aggravation, réévaluation urgente.
"""
print(raw_txt)

Service: Médecine interne  |  Consultation du 17/02/2026 à 09:15
Patient: Mme MARTIN Claire, née le 03/11/1978 (47 ans)
N° dossier:  2026-001928   Tel: 06 12 34 56 78    Email: claire.martin78@exemple.fr

Motif: asthénie + dyspnée d’effort, évoluant depuis ~10 jours.
Contexte: HTA connue; diabète type 2 (metformine). Allergies: NKA.
Traitements habituels: Metformine 850 mg x2/j; Ramipril 5mg 1 cp/j.

Examen clinique:
- TA: 152/96 mmHg   FC: 104/min   SpO2: 94% AA   T°: 38,2°C
- Auscultation: râles crépitants bases bilatérales; pas de sibilants.
- Oedèmes MI: +/-
- Douleur thoracique: non; toux: oui (productive).

Biologie (17/02/2026, 10:40):
CRP  86 mg/L;  Leuco  12,4 G/L; Hb 11.2 g/dL; Plaquettes  410 G/L
Créat  98 µmol/L;  Na  134 mmol/L;  K  3,6 mmol/L;  Glycémie  1,42 g/L
Troponine hs:  9 ng/L (N<14)  |  D-dimères:  780 ng/mL

Imagerie:
RX thorax: opacités alvéolo-interstitielles diffuses (à discuter OAP vs infection).
ECG: tachy sinusale, pas de sus-décalage ST.

Conclusion/Plan:

In [26]:
from __future__ import annotations
import re

_DECIMAL_COMMA = "§DEC_COMMA§"
_DECIMAL_DOT = "§DEC_DOT§"

def normalize_text(raw: str) -> str:
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    text = text.strip()
    text = text.lower()

    lines = text.split("\n")
    lines = [" ".join(line.split()) for line in lines]
    text = "\n".join(lines)

    out_lines = []
    prev_empty = False
    for line in text.split("\n"):
        empty = (line == "")
        if empty and prev_empty:
            continue
        out_lines.append(line)
        prev_empty = empty
    return "\n".join(out_lines)

def clean_for_tokenization(normalized: str) -> str:
    text = normalized

    text = re.sub(r"(\d+),(\d+)", rf"\1{_DECIMAL_COMMA}\2", text)
    text = re.sub(r"(\d+)\.(\d+)", rf"\1{_DECIMAL_DOT}\2", text)

    for ch in [".", ",", ":", ";", "(", ")", "=", "/", "%"]:
        text = text.replace(ch, f" {ch} ")

    text = text.replace(_DECIMAL_COMMA, ",")
    text = text.replace(_DECIMAL_DOT, ".")

    lines = text.split("\n")
    lines = [" ".join(line.split()) for line in lines]
    return "\n".join(lines)

def tokenize_simple(clean: str) -> list[str]:
    flat = clean.replace("\n", " ")
    parts = flat.split(" ")
    return [t for t in parts if t]

def preprocess_note(raw: str) -> dict[str, object]:
    normalized = normalize_text(raw)
    clean = clean_for_tokenization(normalized)
    tokens = tokenize_simple(clean)
    return {
        "raw": raw,
        "normalized": normalized,
        "clean": clean,
        "tokens": tokens,
        "n_tokens": len(tokens),
    }

preprocess_data = preprocess_note(raw_txt)

In [27]:
import re

EMAIL_RE = re.compile(r"\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b", re.IGNORECASE)
DATE_RE = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
TIME_RE = re.compile(r"\b\d{2}:\d{2}\b")

# Datetimes = uniquement les dates qui ont une heure (pas de date seule)
DATETIME_RE = re.compile(r"\b\d{2}/\d{2}/\d{4}\b\s*(?:à|,)\s*\b\d{2}:\d{2}\b")

NUMBER_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b")
PHONE_RE = re.compile(r"\b0[1-9](?:[\s.-]?\d{2}){4}\b")
PATIENT_LINE_RE = re.compile(r"^\s*patient\s*:\s*(.+)$", re.IGNORECASE | re.MULTILINE)


def _dedup_keep_order(items: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in items:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def extract_info(text: str) -> dict[str, object]:
    emails = _dedup_keep_order(EMAIL_RE.findall(text))
    dates = _dedup_keep_order(DATE_RE.findall(text))
    times = _dedup_keep_order(TIME_RE.findall(text))
    datetimes = _dedup_keep_order(DATETIME_RE.findall(text))

    numbers = NUMBER_RE.findall(text)
    phones = _dedup_keep_order(PHONE_RE.findall(text))

    patient_lines = PATIENT_LINE_RE.findall(text)
    names = []
    for pline in patient_lines:
        left = pline.split(",")[0]
        left = left.split("(")[0].strip()
        if left:
            names.append(left.lower())
    names = _dedup_keep_order(names)

    return {
        "emails": emails,
        "dates": dates,
        "times": times,
        "datetimes": datetimes,
        "numbers": numbers,
        "phones": phones,
        "names": names,
    }

extract_info(preprocess_data["normalized"])

{'emails': ['claire.martin78@exemple.fr'],
 'dates': ['17/02/2026', '03/11/1978'],
 'times': ['09:15', '10:40'],
 'datetimes': ['17/02/2026 à 09:15', '17/02/2026, 10:40'],
 'numbers': ['17',
  '02',
  '2026',
  '09',
  '15',
  '03',
  '11',
  '1978',
  '47',
  '2026',
  '001928',
  '06',
  '12',
  '34',
  '56',
  '78',
  '10',
  '2',
  '850',
  '1',
  '152',
  '96',
  '104',
  '94',
  '38,2',
  '17',
  '02',
  '2026',
  '10',
  '40',
  '86',
  '12,4',
  '11.2',
  '410',
  '98',
  '134',
  '3,6',
  '1,42',
  '9',
  '14',
  '780',
  '7',
  '38'],
 'phones': ['06 12 34 56 78'],
 'names': ['mme martin claire']}

In [57]:
from spacy.cli import download

download("fr_core_news_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 11.4 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [28]:
import spacy

def load(model: str = "fr_core_news_md"):
    try:
        nlp = spacy.load(model)
        model_name = model
    except Exception:
        nlp = spacy.blank("fr")
        model_name = "blank_fr"
    return nlp, model_name

In [29]:
nlp, model_name = load()

In [30]:
nlp.pipe_names

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [31]:
text = "Consultation du 17/02/2026. Patient: Mme MARTIN Claire."
doc  = nlp(text)
for t in doc.ents:
    print((t.text, t.label_))

('Mme MARTIN Claire', 'PER')


In [32]:
def run_doc(nlp, text: str):
    return nlp(text)

doc = run_doc(nlp, preprocess_data["normalized"])

In [33]:
def extract_entities(doc):
    ents = []
    for ent in doc.ents:
        ents.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char,
        })
    return ents

In [37]:
def spacy_extract(text: str, model: str = "fr_core_news_md") -> dict[str, object]:
    nlp = spacy.load(model)
    doc = nlp(text)

    entities = [{
        "text": ent.text,
        "label": ent.label_,
        "start": ent.start_char,
        "end": ent.end_char,
    } for ent in doc.ents]

    labels_present = sorted({e["label"] for e in entities})

    allowed = {"PER", "DATE", "TIME", "ORG", "LOC", "GPE"}
    filtered = [e for e in entities if e["label"] in allowed]

    by_label = {}
    for e in filtered:
        by_label.setdefault(e["label"], []).append(e)

    return {
        "model": model,
        "labels_present": labels_present,
        "entities": entities,
        "entities_filtered": filtered,
        "entities_by_label": by_label,
    }

In [42]:
def build_record(raw: str, nlp=None) -> dict[str, object]:
    pre = preprocess_note(raw)

    extracted = extract_info(pre["normalized"])  # regex sur normalized, pas sur clean

    model_name = None
    ner = {"labels_present": [], "entities": [], "entities_filtered": [], "entities_by_label": {}}
    if nlp is not None:
        ner = spacy_extract(pre["normalized"], nlp)
        model_name = getattr(nlp, "meta", {}).get("name", None)

    return {
        "raw": pre["raw"],
        "normalized": pre["normalized"],
        "clean": pre["clean"],
        "tokens": pre["tokens"],
        "n_tokens": pre["n_tokens"],
        "regex": extracted,
        "spacy_model": model_name,
        "ner": ner,
    }